# Practical-equivalence sensitivity analysis

**Humanoid | saved simulated rewards only**

This notebook reads the result file selected by `setup_practical_equivalence.json`; it does not simulate or run the bootstrap. Both plots show effect sizes and **approximate 95% cluster-bootstrap bounds**, compared with exploratory tolerances. Lower values indicate smaller departures under the chosen diagnostic.

An upper bound below a tolerance supports practical equivalence for that measure; a lower bound above it indicates a departure beyond tolerance; overlap is inconclusive. These plots do not certify exact independence or identical distributions. See [method details and limitations](Practical%20equivalence.md).

In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "Statistical tests" / "setup_practical_equivalence.json").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the Cosmic Octopi project.")
folder = root / "Statistical tests"
setup = json.loads((folder / "setup_practical_equivalence.json").read_text())
path = folder / setup["output_file"]
if not path.is_file():
    raise FileNotFoundError("Run Statistical tests/practical_equivalence.py after trajectory generation finishes.")
with path.open("rb") as stream:
    result = pickle.load(stream)
if not result["complete"]:
    raise ValueError("The practical-equivalence analysis is not yet complete; resume its script.")
config = result["configuration"]
horizons = np.asarray(config["horizons"])
plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "stix",
                     "font.size": 12, "axes.labelsize": 13,
                     "axes.titlesize": 14, "figure.dpi": 140, "svg.fonttype": "none"})
display(Markdown(f"**Trajectories:** {result['n_runs']}; **arms:** {len(result['arm_values'])}; "
                 f"**maximum lag:** {config['max_lag']}; "
                 f"**whole-trajectory bootstrap replicates:** {config['bootstrap_replicates']}. "
                 "Confidence bounds are pointwise in horizon and approximate."))

def effect_plot(summary, tolerances, title, ylabel, basename):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.fill_between(horizons, summary["lower"], summary["upper"], color="#215b9c", alpha=.17,
                    label=f"Approximate {config['confidence_level']:.0%} confidence bounds")
    ax.plot(horizons, summary["effect"], color="#215b9c", marker="o", markersize=5,
            linewidth=2, label="Estimated maximum effect")
    for tolerance, color, style in zip(tolerances, ["#30805e", "#d17a16", "#a13d62"], ["--", "-.", ":"]):
        ax.axhline(tolerance, color=color, linestyle=style, linewidth=1.5,
                   label=rf"Tolerance $\delta$ = {tolerance:g}")
    ax.set_xlabel(r"Horizon $p$")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(horizons.min()-2, horizons.max()+2)
    ax.set_xticks([p for p in horizons if p % 20 == 0 or p == horizons[-1]])
    ax.set_ylim(bottom=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)
    ax.legend(frameon=False, fontsize=9, loc="best")
    fig.tight_layout()
    fig.savefig(folder / f"{basename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(folder / f"{basename}.svg", bbox_inches="tight")
    plt.show()


## (a) Practical weak correlation

The effect is the maximum absolute Pearson correlation across the two cost coordinates and lags 1 through 5, pooling lag pairs across trajectories and positions in each prefix. No detrending or removal of initialization is used. Bounds account for the coordinate/lag comparisons within each horizon. This diagnostic can include drift-induced correlation and can miss nonlinear or time-specific dependence: it is not a test establishing independence.

In [ ]:
effect_plot(result["correlation"], config["correlation_tolerances"],
            "(a) Practical weak correlation", "Maximum absolute pooled lag correlation",
            "Practical correlation")

## (b) Practical distributional stability

The effect is the maximum energy distance over **all iteration pairs and all arms** within each prefix. Costs use the original fixed normalization bounds. Bounds address these comparisons jointly within each horizon. The distance is

$$D(F,G)=\sqrt{2\mathbb{E}\|X-Y\|-\mathbb{E}\|X-X'\|-\mathbb{E}\|Y-Y'\|}.$$

The tolerances are in normalized energy-distance units, not percentages of overhead or instability. The empirical V-statistic can be biased upward at small sample sizes, and bootstrap coverage near zero requires further calibration. Broad uncertainty remains inconclusive.

In [ ]:
effect_plot(result["distribution"], config["energy_distance_tolerances"],
            "(b) Practical distributional stability", "Maximum energy distance",
            "Practical distribution")

## Decisions at each exploratory tolerance

These tables distinguish supported equivalence, a departure beyond tolerance, and inconclusive evidence. They use confidence bounds, not failure to reject the original exact-null tests. Horizons overlap and the two diagnostic families are not jointly adjusted.

In [ ]:
for name, summary in [("Pooled lag correlation", result["correlation"]),
                       ("All-arm distributional stability", result["distribution"])]:
    table = {"Horizon": horizons}
    for decision in summary["decisions"]:
        table[f"Tolerance {decision['tolerance']:g}"] = np.where(
            decision["supported"], "Supported", np.where(
                decision["beyond_tolerance"], "Beyond tolerance", "Inconclusive"))
    display(Markdown(f"**{name}**"))
    display(pd.DataFrame(table).set_index("Horizon"))

These are approximate, exploratory bounds; the numerical checks do not establish nominal coverage. Full details, estimands, caveats, and references are in [Practical equivalence.md](Practical%20equivalence.md). No 20% rejection-rate cutoff is applied here: effect tolerances answer a different question.